# Build a served, tested and monitored ticket classifier

Train a small text classifier for synthetic support tickets. It may recommend a standard
or urgent queue or ask a person to review. It cannot close cases, modify tickets or make
decisions outside the support team's scope. Freeze a versioned artifact, use it for request
handling, evaluate on unseen ticket families, write a drift report and rehearse rollback.
Then package it: a FastAPI service in a local Docker container, pytest tests, a GitHub
Actions workflow and an MLflow run.

Sections 1 to 6 (the offline core) use Python's standard library only and run in local
Jupyter or Colab. Section 7 writes real project files into a `capstone/` folder next to this
notebook and section 8 records an MLflow run; both need the pinned packages from the install
cell. Docker and GitHub Actions run on your own machine and GitHub account, not in Colab.
Fill the TODOs and write your final deployment recommendation.


In [ ]:
import hashlib
import json
import math
import re
from collections import Counter

# A deliberately small fixture. Text in each split belongs to different ticket families.
TRAIN = [
    ("device-down", "outage blocks all devices", 1),
    ("checkout-down", "checkout outage blocks purchases", 1),
    ("account-lockout", "account inaccessible blocks login", 1),
    ("gateway", "production outage customer access blocked", 1),
    ("invoice-copy", "invoice copy request receipt", 0),
    ("account-name", "profile account name change request", 0),
    ("manual", "documentation request for device guide", 0),
    ("export", "request billing export receipt", 0),
]
VALIDATION = [
    ("checkout-slow", "checkout blocked in production", 1),
    ("fleet", "outage blocks fleet access", 1),
    ("profile-photo", "profile photo change request", 0),
    ("billing-date", "invoice request for receipt date", 0),
    ("login-name", "account login name change", 1),
    ("device-blocked", "device request blocked", 0),
]
TEST = [
    ("site-down", "site outage blocks production", 1, "web"),
    ("tablet", "device inaccessible and checkout blocked", 1, "mobile"),
    ("tax-receipt", "request invoice receipt for taxes", 0, "web"),
    ("display-name", "profile display name change request", 0, "mobile"),
    ("rare-critical", "unusual blank screen", 1, "mobile"),
]
assert len({family for family, *_ in TRAIN+VALIDATION+TEST}) == len(TRAIN)+len(VALIDATION)+len(TEST)

def words(text):
    return re.findall(r"[a-z]+", text.casefold())

def digest(value):
    canonical = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(canonical.encode()).hexdigest()

def fit_model(rows):
    counts = [Counter(), Counter()]
    priors = [0, 0]
    for _, text, label in rows:
        priors[label] += 1
        counts[label].update(words(text))
    vocabulary = sorted(counts[0].keys() | counts[1].keys())
    return {"vocabulary":vocabulary, "counts":[dict(row) for row in counts],
            "totals":[sum(row.values()) for row in counts], "class_counts":priors}

def predict_probability(text, model):
    tokens = [token for token in words(text) if token in model["vocabulary"]]
    if not tokens:
        return None
    scores = []
    vocab_size = len(model["vocabulary"])
    for label in (0, 1):
        total = model["totals"][label] + vocab_size
        count = model["counts"][label]
        scores.append(math.log(model["class_counts"][label]) +
                      sum(math.log((count.get(token, 0)+1)/total) for token in tokens))
    maximum = max(scores)
    weights = [math.exp(score-maximum) for score in scores]
    return weights[1] / sum(weights)

def route(probability, lower, upper):
    if probability is None:
        return "review"
    if probability <= lower:
        return "standard"
    return "urgent" if probability >= upper else "review"

model = fit_model(TRAIN)
assert predict_probability("outage blocks", model) > predict_probability("invoice request", model)
assert predict_probability("snowflake", model) is None


## 1. Choose thresholds on validation

Implement validation_score(rows, model, lower, upper). Count false-standard, false-urgent and
review decisions. Use the explicit fixture costs: false standard 5, false urgent 2, review 1,
correct automatic decision 0. Return (cost, review_count). The supplied selection cell uses
validation only. In your report explain what could make the relative costs inappropriate. The Naive Bayes score is not assumed to be calibrated.


In [ ]:
def validation_score(rows, model, lower, upper):
    raise NotImplementedError


In [ ]:
assert validation_score([("a","snowflake",1)],model,0.2,0.8) == (1,1)
assert validation_score([("a","invoice receipt request",1)],model,0.49,0.51) == (5,0)
assert validation_score([("a","production outage blocks",0)],model,0.49,0.51) == (2,0)

print("OK")


In [ ]:
BANDS = [(0.15,0.85),(0.25,0.75),(0.4,0.6)]
chosen = min(BANDS, key=lambda band: (validation_score(VALIDATION,model,*band), band))
manifest = {"schema_version":1, "training_digest":digest(TRAIN), "validation_digest":digest(VALIDATION),
            "preprocessor":"ascii-words-v1", "thresholds":list(chosen), "model":model}
version = digest(manifest)
artifact = {**manifest, "version":version}
# Round-trip the artifact through exactly the shape a service can load from disk.
candidate = json.loads(json.dumps(artifact,sort_keys=True))
assert digest({key:value for key,value in candidate.items() if key != "version"}) == candidate["version"]
print("Selected validation band:", chosen, "artifact:", version[:12])


## 2. Serve an explicit prediction contract

Implement predict_request(payload, artifact). Accept only a dictionary with exactly ticket and
channel. Ticket must be a nonblank string no longer than 500 characters and channel must be web
or mobile. Invalid inputs return (400, {"error":"invalid_input"}). Valid inputs return a 200
status and a dictionary with decision, probability, and model_version. Read the thresholds and
model from the artifact argument, never from global variables. Unknown vocabulary can
produce None and a review outcome. Never echo the ticket. Section 7 copies this
function into a module, so use only `words`, `predict_probability`, `route` and the standard library.


In [ ]:
def predict_request(payload, artifact):
    raise NotImplementedError


In [ ]:
assert predict_request({"ticket":" ","channel":"web"},candidate) == (400,{"error":"invalid_input"})
assert predict_request({"ticket":"x"*501,"channel":"web"},candidate)[0] == 400
assert predict_request({"ticket":"invoice","channel":[]},candidate)[0] == 400
assert predict_request({"ticket":"invoice","channel":"web","id":"extra"},candidate)[0] == 400
status, body = predict_request({"ticket":"outage blocks production","channel":"mobile"},candidate)
assert status == 200 and body["decision"] == "urgent" and body["model_version"] == version
assert "ticket" not in body
assert predict_request({"ticket":"snowflake","channel":"web"},candidate)[1]["decision"] == "review"
assert predict_request(["ticket"],candidate)[0] == 400
assert predict_request({"ticket":5,"channel":"web"},candidate)[0] == 400
assert predict_request({"ticket":"invoice"},candidate)[0] == 400
# The thresholds must come from the artifact passed in, not from a global variable.
assert predict_request({"ticket":"outage blocks production","channel":"web"},{**candidate,"thresholds":[0.99,0.995]})[1]["decision"] == "standard"

print("OK")


## 3. Monitor decisions and delayed outcomes

Implement monitor(rows). Each row contains (channel, decision, outcome). Outcome is 0, 1 or
None before a label arrives. Return counts by channel for attempted, reviewed, unlabelled,
known positive and missed urgent. Here a missed urgent is a known positive routed standard.
Positive reviews are tracked as review volume and are not counted as missed automatic cases.
The workflow still needs reviewer SLA measurement.


In [ ]:
def monitor(rows):
    raise NotImplementedError


In [ ]:
rows = [("web","review",None),("web","standard",1),("web","review",1),("mobile","urgent",0)]
assert monitor(rows) == {"web":{"attempted":3,"reviewed":2,"unlabelled":1,"positive":2,"missed_urgent":1},
                         "mobile":{"attempted":1,"reviewed":0,"unlabelled":0,"positive":0,"missed_urgent":0}}
assert monitor([]) == {}

print("OK")


## 4. Run the final holdout after fixing the operating policy

Before running this cell, record how to handle an urgent miss, missing labels and review
overload. Name the model owner and the previous safe route. The toy holdout labels are visible
for teaching; use a separate evaluator and immutable labels in a real deployment. Do not
adjust thresholds after viewing these scores.


In [ ]:
scored = []
for _, text, label, channel in TEST:
    status, body = predict_request({"ticket":text,"channel":channel},candidate)
    assert status == 200
    scored.append((channel,body["decision"],label))

baseline = [(channel,"standard",label) for _, _, label, channel in TEST]
final_counts, baseline_counts = monitor(scored), monitor(baseline)
print("Candidate:", final_counts)
print("Always-standard baseline:", baseline_counts)
assert sum(item["attempted"] for item in final_counts.values()) == len(TEST)


## 5. Exercise the release stop and fallback

Implement next_release(summary, minimum_labelled, max_review_fraction, critical_incident).
Roll back immediately on a critical incident or any known missed urgent ticket. Otherwise
hold for zero traffic, inadequate labelled cases, or review fraction above the limit.
Promote only if every other condition passes. This deliberately strict gate is for the
exercise. Zero known positives is missing quality evidence and should also hold.


In [ ]:
def next_release(summary, minimum_labelled, max_review_fraction, critical_incident):
    raise NotImplementedError


In [ ]:
assert next_release({},1,0.2,True) == "rollback"
assert next_release(monitor([("web","standard",1)]),100,0.2,False) == "rollback"
assert next_release({},0,1,False) == "hold"
assert next_release(monitor([("web","standard",0)]),1,1,False) == "hold"
assert next_release(monitor([("web","urgent",1)]),2,1,False) == "hold"
assert next_release(monitor([("web","review",1)]),1,0.5,False) == "hold"
assert next_release(monitor([("web","urgent",1),("web","review",None)]),1,0.5,False) == "promote"
print("Holdout gate:", next_release(final_counts,10,0.5,False))

print("OK")


## 6. Write a drift report

Labels arrive late, but the input mix can be compared immediately. Reuse the total variation
idea from A10 ([measure input drift](/mle/a10/measure-input-drift/)).

1. Implement `total_variation(reference, current)` for count dictionaries: normalise each to
   proportions, take half the sum of absolute differences over the union of categories, and
   return None if either total is zero.
2. Implement `drift_report(reference, current, model, threshold)`. Both windows are lists of
   request dictionaries with ticket and channel. For each window build two histograms: channel
   counts, and a vocabulary histogram with keys familiar and unfamiliar (unfamiliar when
   `predict_probability` returns None). Return a dictionary with `reference_count`,
   `current_count`, `channel_tv`, `unfamiliar_tv` and `alert`, which is true when either
   distance is not None and is strictly above the threshold.

A single window is an investigation signal, not proof of harm: request mix, preprocessing or
system behaviour could also have changed. A production alert would also require persistence
across several valid windows, as in A10. Section 7 saves your report as `drift_report.json`.


In [ ]:
def total_variation(reference, current):
    raise NotImplementedError

def drift_report(reference, current, model, threshold):
    raise NotImplementedError


In [ ]:
reference_requests = [{"ticket":text,"channel":channel} for _, text, _, channel in TEST]
current_requests = [{"ticket":"invoice request","channel":"web"},
                    {"ticket":"glitched flicker","channel":"mobile"},
                    {"ticket":"snowflake","channel":"mobile"},
                    {"ticket":"outage blocks checkout","channel":"mobile"}]
assert abs(total_variation({"a":8,"b":2},{"a":5,"b":5})-0.3) < 1e-12
assert total_variation({"a":2},{"b":3}) == 1
assert total_variation({"a":1},{}) is None
report = drift_report(reference_requests, current_requests, candidate["model"], 0.25)
assert report["reference_count"] == 5 and report["current_count"] == 4
assert abs(report["channel_tv"]-0.15) < 1e-12 and abs(report["unfamiliar_tv"]-0.3) < 1e-12
assert report["alert"] == True
assert drift_report(reference_requests, current_requests, candidate["model"], 0.35)["alert"] == False
assert drift_report(reference_requests, reference_requests, candidate["model"], 0.25)["alert"] == False
empty = drift_report(reference_requests, [], candidate["model"], 0.25)
assert empty["current_count"] == 0 and empty["channel_tv"] is None and empty["unfamiliar_tv"] is None
assert empty["alert"] == False
print("Drift report:", report)
print("Pending labels:", monitor([("mobile","review",None),("web","urgent",None)]))

print("OK")


## 7. Package it

These cells write a small service project into `capstone/`. Run them in order; rerunning
overwrites the files. Install the pinned packages first (versions as of late 2026; upgrade
deliberately and rerun the tests). In local Jupyter, run the notebook's kernel from a
virtual environment, for example `uv venv` then `uv pip install jupyter`.

The service reuses your offline code: the next cell copies `words`, `predict_probability`,
`route` and your `predict_request` into `core.py` and saves the frozen artifact as
`model.json`, so the container serves exactly the version you evaluated.


In [ ]:
%pip install -q fastapi==0.162.0 starlette==1.10.0 pydantic==2.14.2 uvicorn==0.54.0 httpx2==2.16.0 pytest==9.2.0 mlflow==3.17.2


In [ ]:
import inspect
from pathlib import Path

CAPSTONE = Path("capstone")

def write(path, text):
    target = CAPSTONE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text)
    print("wrote", target)

write("core.py", "import hashlib\nimport json\nimport math\nimport re\n\n\n" +
      "\n\n".join(inspect.getsource(function)
                  for function in (words, digest, predict_probability, route, predict_request)))
write("model.json", json.dumps(candidate, sort_keys=True, indent=2) + "\n")
write("drift_report.json", json.dumps(report, indent=2) + "\n")


In [ ]:
write("app.py", """import json
from pathlib import Path
from typing import Literal

from fastapi import FastAPI, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from pydantic import BaseModel, ConfigDict

from core import digest, predict_request

ARTIFACT = json.loads((Path(__file__).parent / "model.json").read_text())
if digest({key: value for key, value in ARTIFACT.items() if key != "version"}) != ARTIFACT["version"]:
    raise RuntimeError("model.json does not match its recorded version")

app = FastAPI(title="ticket-clf")


class TicketRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    ticket: str
    channel: Literal["web", "mobile"]


@app.exception_handler(RequestValidationError)
async def invalid_input(request: Request, exc: RequestValidationError):
    # FastAPI's default 422 body echoes the input. The contract is a 400 that never echoes the ticket.
    return JSONResponse({"error": "invalid_input"}, status_code=400)


@app.get("/health")
def health():
    return {"status": "ok", "model_version": ARTIFACT["version"]}


@app.post("/predict")
def predict(request: TicketRequest):
    # Blank and overlong tickets are still rejected by predict_request, the single contract.
    status, body = predict_request(request.model_dump(), ARTIFACT)
    return JSONResponse(body, status_code=status)
""")


In [ ]:
write("requirements.txt", """# Runtime pins, as of late 2026. Upgrade deliberately and rerun the tests.
fastapi==0.162.0
starlette==1.10.0
pydantic==2.14.2
uvicorn==0.54.0
""")
write("requirements-dev.txt", """# Test-only pins. Starlette's TestClient uses httpx2.
-r requirements.txt
httpx2==2.16.0
pytest==9.2.0
""")
write("pytest.ini", """[pytest]
pythonpath = .
testpaths = tests
""")


In [ ]:
write("Dockerfile", """# Slim base pinned to an exact Python release (as of late 2026).
FROM python:3.14.9-slim-trixie

ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY core.py app.py model.json ./

RUN useradd --create-home --uid 10001 appuser
USER appuser

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
""")


In [ ]:
write("tests/test_app.py", """from fastapi.testclient import TestClient

from app import ARTIFACT, app
from core import predict_request

client = TestClient(app)
VERSION = ARTIFACT["version"]


def test_health():
    reply = client.get("/health")
    assert reply.status_code == 200
    assert reply.json() == {"status": "ok", "model_version": VERSION}


def test_predict_urgent():
    reply = client.post("/predict", json={"ticket": "outage blocks production", "channel": "mobile"})
    body = reply.json()
    assert reply.status_code == 200
    assert body["decision"] == "urgent" and body["model_version"] == VERSION
    assert "ticket" not in body


def test_http_rejects_bad_requests_without_echo():
    for payload in [{"ticket": "invoice", "channel": "web", "id": "extra"},
                    {"ticket": "invoice"},
                    {"ticket": 5, "channel": "web"},
                    {"ticket": " ", "channel": "web"},
                    {"ticket": "x" * 501, "channel": "web"},
                    ["ticket"]]:
        reply = client.post("/predict", json=payload)
        assert reply.status_code == 400
        assert reply.json() == {"error": "invalid_input"}


def test_handler_contract():
    candidate = ARTIFACT
    assert predict_request({"ticket": " ", "channel": "web"}, candidate) == (400, {"error": "invalid_input"})
    assert predict_request({"ticket": "x" * 501, "channel": "web"}, candidate)[0] == 400
    assert predict_request({"ticket": "invoice", "channel": []}, candidate)[0] == 400
    assert predict_request({"ticket": "invoice", "channel": "web", "id": "extra"}, candidate)[0] == 400
    status, body = predict_request({"ticket": "outage blocks production", "channel": "mobile"}, candidate)
    assert status == 200 and body["decision"] == "urgent" and body["model_version"] == VERSION
    assert "ticket" not in body
    assert predict_request({"ticket": "snowflake", "channel": "web"}, candidate)[1]["decision"] == "review"
    assert predict_request(["ticket"], candidate)[0] == 400
    assert predict_request({"ticket": 5, "channel": "web"}, candidate)[0] == 400
    assert predict_request({"ticket": "invoice"}, candidate)[0] == 400
    stricter = {**candidate, "thresholds": [0.99, 0.995]}
    assert predict_request({"ticket": "outage blocks production", "channel": "web"}, stricter)[1]["decision"] == "standard"
""")


In [ ]:
write(".github/workflows/ci.yml", """name: ci

on:
  push:
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v6
      - uses: actions/setup-python@v6
        with:
          python-version: "3.14"
          cache: pip
          cache-dependency-path: requirements-dev.txt
      - run: pip install -r requirements-dev.txt
      - run: python -m pytest -q
      - run: docker build -t ticket-clf .
""")


In [ ]:
import importlib.util
import subprocess
import sys

expected = ["core.py", "model.json", "drift_report.json", "app.py", "requirements.txt",
            "requirements-dev.txt", "pytest.ini", "Dockerfile", "tests/test_app.py",
            ".github/workflows/ci.yml"]
missing = [path for path in expected if not (CAPSTONE / path).is_file()]
assert not missing, f"Rerun the cells above; missing {missing}"
for name in ("requirements.txt", "requirements-dev.txt"):
    pins = [line for line in (CAPSTONE / name).read_text().splitlines()
            if line and not line.startswith(("#", "-r"))]
    assert pins and all("==" in line for line in pins), f"Pin every package in {name}"
assert "USER appuser" in (CAPSTONE / "Dockerfile").read_text()
assert json.loads((CAPSTONE / "model.json").read_text())["version"] == version
assert all(importlib.util.find_spec(name) for name in ("fastapi", "httpx2", "pytest")), \
    "Run the install cell above, restart the kernel if asked, then rerun this check"
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=CAPSTONE,
                        capture_output=True, text=True)
print(result.stdout[-1500:])
assert result.returncode == 0, result.stderr[-1500:]

print("OK")


### Run the container and CI

Docker and GitHub Actions are not available in Colab. Download the `capstone/` folder if you
worked there, then on your own machine with Docker running:

```bash
cd capstone
docker build -t ticket-clf . && docker run -p 8000:8000 ticket-clf
```

In a second terminal:

```bash
curl http://127.0.0.1:8000/health
curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d '{"ticket": "outage blocks production", "channel": "web"}'
```

Check that `model_version` matches the artifact version printed in section 1, and that a
request with an extra field returns 400. To see CI run, make `capstone/` the root of a new
GitHub repository (`git init`, commit, add the remote, push). The workflow runs the tests and
builds the image on every push and pull request. Record the run link in your deployment note.


## 8. Track the run in MLflow

Record the selected band, the validation cost, the holdout counts and the frozen artifact in
a local MLflow store: `mlflow.db` plus an `mlruns/` folder for artifacts, both next to this
notebook and outside `capstone/`. Afterwards, from this folder run
`mlflow ui --backend-store-uri sqlite:///mlflow.db` and open the printed address.


In [ ]:
import mlflow

# The store sits next to this notebook, outside capstone/, so it is never committed or copied into the image.
mlflow.set_tracking_uri("sqlite:///" + str(Path("mlflow.db").resolve()))
mlflow.set_experiment("ticket-classifier-capstone")
with mlflow.start_run(run_name=version[:12]) as run:
    mlflow.log_params({"lower": chosen[0], "upper": chosen[1], "model_version": version,
                       "preprocessor": manifest["preprocessor"]})
    mlflow.log_metric("validation_cost", validation_score(VALIDATION, model, *chosen)[0])
    for channel, counts in final_counts.items():
        for name, value in counts.items():
            mlflow.log_metric(f"holdout_{channel}_{name}", value)
    mlflow.log_artifact(str(CAPSTONE / "model.json"))
    mlflow.log_artifact(str(CAPSTONE / "drift_report.json"))

logged = mlflow.get_run(run.info.run_id)
assert logged.data.params["lower"] == str(chosen[0]) and "holdout_web_attempted" in logged.data.metrics
print("MLflow run", run.info.run_id, "recorded for artifact", version[:12])


## Submit a deployment note

- Record the artifact version, intended use, out-of-scope decisions, data limitations and owner.
- Compare the candidate and always-standard baseline by channel. Report attempts, reviews, missing
  labels and known missed urgencies. Explain why the fixture is insufficient to authorise deployment.
- Rehearse a rollback and name the alternate route, including who owns the review queue and its SLA.
- Summarise your drift report and name an outcome-based signal. Explain how delayed labels affect each.
- Record your validation threshold, release rule, the MLflow run ID, whether the Docker image
  built and served /predict locally, and the link to a passing CI run.
